# Import package

In [1]:
import torch
import os
from utils.protein_init import *
from utils.RNA_init import *
from utils.RP_init import *
from utils.dataset import *
import pandas as pd
from utils.Predictor import Predictor, calculate_performance 
from utils.utils import DataLoader

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


# Load model and data

In [2]:
### Load model 
model_path = f'trained_model/model.pth'
RNA_thresh = 0.521667
data_path = 'Data/'
train_mode = 'struc'

In [3]:
# Load the entire model (including structure and parameters)
model = torch.load(
    model_path,
    map_location=device,
)

model = model.to(device)
model.device = torch.device(device)
model.eval()


net(
  (RNA_evo): MLP(
    (FC_layers): ModuleList(
      (0): Linear(in_features=640, out_features=400, bias=True)
      (1): Linear(in_features=400, out_features=200, bias=True)
    )
    (out_ln): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
  )
  (RNA_aa): MLP(
    (FC_layers): ModuleList(
      (0): Linear(in_features=21, out_features=400, bias=True)
      (1): Linear(in_features=400, out_features=200, bias=True)
    )
    (out_ln): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
  )
  (prot_evo): MLP(
    (FC_layers): ModuleList(
      (0): Linear(in_features=1280, out_features=400, bias=True)
      (1): Linear(in_features=400, out_features=200, bias=True)
    )
    (out_ln): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
  )
  (prot_aa): MLP(
    (FC_layers): ModuleList(
      (0): Linear(in_features=33, out_features=400, bias=True)
      (1): Linear(in_features=400, out_features=200, bias=True)
    )
    (out_ln): LayerNorm((200,), eps=1e-05, elementwise_a

In [4]:
### Load data
test_file = f"{data_path}FoldBench21_set.csv"

test_df = pd.read_csv(test_file)

pdbcodes = test_df['PDB_code'].tolist()

RNA_path = os.path.join(f'{data_path}', f'FoldBench21_set_RNA_fea.pt')
RNA_dict = torch.load(RNA_path)

protein_path = os.path.join(f'{data_path}', f'FoldBench21_set_protein_fea.pt')
prot_dict = torch.load(protein_path)


RP_path = os.path.join(f'{data_path}', f'FoldBench21_label.pt')


RP_conmap_truth_dict = torch.load(RP_path)

print(test_df)
test_dataset = RNAProteinMoleculeDataset(test_df, RNA_dict, prot_dict, RP_conmap_truth_dict, device=device)
print(test_dataset)

test_loader = DataLoader(test_dataset, batch_size=1, follow_batch=['RNA_node_aa', 'prot_node_aa'])

FileNotFoundError: [Errno 2] No such file or directory: 'Data/FoldBench21_set.csv'

# Define Predictor

In [ ]:
Predictor = Predictor(model=model)

# Evaluation

In [ ]:
save_path = f'result/FoldBench21'
os.makedirs(save_path, exist_ok=True)

result_dfs = pd.DataFrame(columns=['pdbname','RNA_pre', 'RNA_rec', 'RNA_f1'])
# print(result_dfs)
for data in test_loader:
    result_df = {}
    # print(data.pdbname)
    data = data.to(device)
    test_id = data.pdbname[0]
    
    RNA_final_scores = Predictor.predict(data=data)
    # print(RNA_final_scores)
    RNA_final_scores = RNA_final_scores[0]
    
    RNA_final_scores_cpu = RNA_final_scores.cpu().numpy()
    pred_score_df = pd.DataFrame(RNA_final_scores_cpu, columns=['pred_score'])
    
    RNA_pred_labels = (RNA_final_scores_cpu >= RNA_thresh).astype(int)
    
    pred_df = pd.DataFrame(RNA_pred_labels, columns=['pred_label'])
    pred_df = pd.concat([pred_score_df, pred_df], axis=1)


    pred_df_file = f'./result/FoldBench21/{test_id}.csv'
    pred_df.to_csv(pred_df_file, index=False)

    RNA_site_truth = data.RNA_site_truth.squeeze(-1) # [N, 1] -> [N]
    # print(RNA_site_truth)
    rna_site_pre, rna_site_rec, rna_site_f1 \
    = calculate_performance(RNA_final_scores, RNA_site_truth, data.RNA_node_aa_batch, threshold=RNA_thresh)

    ### write result ###
    result_df['pdbname'] = data.pdbname[0]
    result_df['RNA_pre'] = rna_site_pre
    result_df['RNA_rec'] = rna_site_rec
    result_df['RNA_f1'] = rna_site_f1

    result_df = pd.DataFrame.from_dict(result_df, orient='index').T
    
    result_dfs = pd.concat([result_dfs, result_df], axis=0)

df_long = result_dfs[['pdbname', 'RNA_pre', 'RNA_rec', 'RNA_f1']]\
.melt(id_vars='pdbname', var_name='Performance', value_name='Value')
df_long['Methods'] = 'ZHMolSite'

save_file = f'{save_path}/RNA_Pre_Rec_F1.csv'
df_long.to_csv(save_file, index=False)

In [ ]:

print(
    f"RNA Performance (Average)\n"
    f"----------------------\n"
    f"Precision : {result_dfs['RNA_pre'].mean():.3f}\n"
    f"Recall    : {result_dfs['RNA_rec'].mean():.3f}\n"
    f"F1-score  : {result_dfs['RNA_f1'].mean():.3f}"
)